# OpenPathAI — Quick Start

A single-notebook tour of the Phase 2–5 surface. Runs on any CPU
(laptop, Colab), no dataset download, no torch required.

You'll:

1. Inspect the shipped dataset + model registries (Phase 2 + 3).
2. Load and execute a pipeline YAML (Phase 5).
3. Round-trip the resulting manifest + artifacts through the
   content-addressable cache (Phase 1).

## 1. Registries — what's available out of the box

In [ ]:
from openpathai import default_model_registry, default_registry

print('Datasets:', default_registry().names())
print('Models:  ', default_model_registry().names())

Both registries are discovered from on-disk YAML cards under
`data/datasets/` and `models/zoo/`. Users can drop additional cards
into `~/.openpathai/datasets/` and `~/.openpathai/models/` without
touching the repo.

## 2. Load + execute a pipeline YAML

In [ ]:
from pathlib import Path

from openpathai import (
    ContentAddressableCache,
    Executor,
    load_pipeline,
)

pipeline = load_pipeline(Path('..') / 'pipelines' / 'supervised_synthetic.yaml')
print(pipeline.id, '->', [step.id for step in pipeline.steps])

In [ ]:
import tempfile

cache_root = Path(tempfile.mkdtemp(prefix='openpathai-'))
cache = ContentAddressableCache(root=cache_root)
executor = Executor(cache)

first = executor.run(pipeline)
print(f'hits={first.cache_stats.hits}  misses={first.cache_stats.misses}')
for step_id, artifact in first.artifacts.items():
    print(f'  {step_id}: {artifact.artifact_type} value={getattr(artifact, "value", "?")}')


## 3. Cache hits — same pipeline, second run

In [ ]:
second = executor.run(pipeline)
print(f'hits={second.cache_stats.hits}  misses={second.cache_stats.misses}')

Identical inputs → zero node invocations. Every step's output is
keyed by `sha256(node_id + code_hash + canonical_json(input_config)
+ canonical_json(sorted(upstream_hashes)))`, so even trivial
rearrangements of the YAML hash identically.

## 4. Next steps

* `openpathai train --model resnet18 --num-classes 4 --synthetic` —
  runs the Phase 3 training engine end-to-end (requires `[train]`).
* `openpathai analyse --model resnet18 --tile TILE.png --explainer gradcam --target-layer layer4`
  — generates a Grad-CAM heatmap.
* `openpathai datasets list` + `openpathai download DATASET --yes` —
  stage a real cohort (review size warnings first).

See [docs/cli.md](../docs/cli.md) for the full subcommand reference.